# Project 15: Movie Theater Box Office & Concession Analysis
**Dataset:** `15_cinema_sales.csv`  
Analysis of ticket sales, showtimes, genres and snack-bar purchases to help optimise screen allocation and combo promotions.

In [4]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

CSV_PATH = "15_cinema_sales.csv"   # change path if the file is elsewhere

## Task 1. Load and inspect the dataset

In [5]:
df = pd.read_csv(CSV_PATH)
print("Shape:", df.shape)
print("\nFirst 5 rows:\n", df.head())
print("\nInfo:")
df.info()
print("\nSummary statistics:\n", df.describe())

Shape: (710, 10)

First 5 rows:
   Transaction_ID   Show_Date       Movie_Title      Genre Showtime_Slot Screen_Format  Tickets_Quantity  Ticket_Price  Occupancy_Percentage  Concession_Spend
0         C00000  2026-10-13  The Last Journey  Animation    Late Night   Standard 2D                 4        154.21                  26.8            557.44
1         C00001  2025-07-05       Hidden City     Action    Late Night            3D                 6        182.11                  67.1               NaN
2         C00002  2025-05-02    Laugh Out Loud     Action    Late Night            3D                 2        287.64                  47.6            544.21
3         C00003  2025-08-05  The Last Journey     Sci-Fi    Late Night   Standard 2D                 1        432.39                  69.9            452.22
4         C00004  2025-04-27      Ocean Planet     Comedy       Matinee   Standard 2D                 6        224.63                  64.5            355.05

Info:
<class 'pand

## Task 2. Missing values and duplicate Transaction_ID records

In [6]:
print("Missing values per column:\n", df.isna().sum())

dup_mask = df["Transaction_ID"].duplicated(keep=False)
print("\nRows sharing a Transaction_ID (all copies):", dup_mask.sum())
print("Extra duplicate rows (beyond first occurrence):",
      df["Transaction_ID"].duplicated().sum())
print("Fully identical duplicate rows:", df.duplicated().sum())
print("\nDuplicate records:\n",
      df[dup_mask].sort_values("Transaction_ID").head(20))

# Remove duplicate Transaction_IDs (keep first occurrence)
df = df.drop_duplicates(subset="Transaction_ID", keep="first").reset_index(drop=True)
print("\nShape after removing duplicate Transaction_IDs:", df.shape)

Missing values per column:
 Transaction_ID           0
Show_Date                0
Movie_Title              0
Genre                    0
Showtime_Slot            0
Screen_Format            0
Tickets_Quantity         0
Ticket_Price             0
Occupancy_Percentage     0
Concession_Spend        17
dtype: int64

Rows sharing a Transaction_ID (all copies): 20
Extra duplicate rows (beyond first occurrence): 10
Fully identical duplicate rows: 10

Duplicate records:
     Transaction_ID   Show_Date       Movie_Title      Genre Showtime_Slot Screen_Format  Tickets_Quantity  Ticket_Price  Occupancy_Percentage  Concession_Spend
155         C00155  2025-07-25    Laugh Out Loud     Comedy       Evening          IMAX                 1        191.39                  55.1            141.99
703         C00155  2025-07-25    Laugh Out Loud     Comedy       Evening          IMAX                 1        191.39                  55.1            141.99
158         C00158  2025-12-22      Robot Reborn  Anim

## Task 3. Clean missing Concession_Spend (justified strategy)

In [7]:
missing_n = df["Concession_Spend"].isna().sum()
print("Missing Concession_Spend:", missing_n)
print("Skewness:", round(df["Concession_Spend"].skew(), 3),
      "| Mean:", round(df["Concession_Spend"].mean(), 2),
      "| Median:", round(df["Concession_Spend"].median(), 2))

# Strategy: fill with the MEDIAN spend of the same Screen_Format + Showtime_Slot
# group (spending habits differ by format/time), falling back to the overall
# median. The median is robust to outliers, and group-level imputation keeps
# segment averages from being distorted. Filling with 0 would wrongly assume
# "no purchase" and understate the attach rate.
group_median = df.groupby(["Screen_Format", "Showtime_Slot"])["Concession_Spend"] \
                 .transform("median")
df["Concession_Spend"] = (df["Concession_Spend"]
                          .fillna(group_median)
                          .fillna(df["Concession_Spend"].median()))
print("Missing after cleaning:", df["Concession_Spend"].isna().sum())

Missing Concession_Spend: 17
Skewness: -0.054 | Mean: 351.65 | Median: 356.88
Missing after cleaning: 0


## Task 4. Convert Show_Date to datetime; weekday vs weekend indicators

In [8]:
df["Show_Date"] = pd.to_datetime(df["Show_Date"], errors="coerce")
print("Unparseable dates:", df["Show_Date"].isna().sum())

df["Weekday_Name"] = df["Show_Date"].dt.day_name()
df["Day_Of_Week"] = df["Show_Date"].dt.dayofweek          # Mon=0 ... Sun=6
df["Is_Weekend"] = np.where(df["Day_Of_Week"] >= 5, 1, 0)  # Sat/Sun = 1
df["Day_Type"] = np.where(df["Is_Weekend"] == 1, "Weekend", "Weekday")
print(df[["Show_Date", "Weekday_Name", "Is_Weekend", "Day_Type"]].head())
print("\n", df["Day_Type"].value_counts())

Unparseable dates: 0
   Show_Date Weekday_Name  Is_Weekend Day_Type
0 2026-10-13      Tuesday           0  Weekday
1 2025-07-05     Saturday           1  Weekend
2 2025-05-02       Friday           0  Weekday
3 2025-08-05      Tuesday           0  Weekday
4 2025-04-27       Sunday           1  Weekend

 Day_Type
Weekday    511
Weekend    189
Name: count, dtype: int64


## Task 5. Total ticket revenue, concession revenue, combined revenue

In [9]:
df["Ticket_Revenue"] = df["Tickets_Quantity"] * df["Ticket_Price"]
df["Total_Spend"] = df["Ticket_Revenue"] + df["Concession_Spend"]

total_ticket_rev = df["Ticket_Revenue"].sum()
total_conc_rev = df["Concession_Spend"].sum()
combined_rev = df["Total_Spend"].sum()
print(f"Total ticket revenue     : {total_ticket_rev:,.2f}")
print(f"Total concession revenue : {total_conc_rev:,.2f}")
print(f"Combined revenue         : {combined_rev:,.2f}")
print(f"Concession share of total: {total_conc_rev / combined_rev:.1%}")

Total ticket revenue     : 867,210.99
Total concession revenue : 246,311.80
Combined revenue         : 1,113,522.79
Concession share of total: 22.1%


## Task 6. Top 10 movies by total ticket sales and ticket revenue

In [10]:
movies = (df.groupby("Movie_Title")
            .agg(Total_Tickets=("Tickets_Quantity", "sum"),
                 Ticket_Revenue=("Ticket_Revenue", "sum"))
            .round(2))
# Note: the dataset has only a handful of distinct movies, so fewer than 10 rows appear.
print("Top 10 by tickets sold:\n",
      movies.sort_values("Total_Tickets", ascending=False).head(10))
print("\nTop 10 by ticket revenue:\n",
      movies.sort_values("Ticket_Revenue", ascending=False).head(10))

Top 10 by tickets sold:
                   Total_Tickets  Ticket_Revenue
Movie_Title                                    
The Last Journey            592       182989.09
Galaxy Quest                471       150069.41
Laugh Out Loud              463       138050.46
Ocean Planet                437       137501.81
Robot Reborn                429       127760.79
Hidden City                 423       130839.43

Top 10 by ticket revenue:
                   Total_Tickets  Ticket_Revenue
Movie_Title                                    
The Last Journey            592       182989.09
Galaxy Quest                471       150069.41
Laugh Out Loud              463       138050.46
Ocean Planet                437       137501.81
Hidden City                 423       130839.43
Robot Reborn                429       127760.79


## Task 7. Average occupancy % across screen formats

In [11]:
fmt = (df.groupby("Screen_Format")[["Occupancy_Percentage", "Tickets_Quantity",
                                     "Ticket_Price", "Concession_Spend"]]
         .mean(numeric_only=True).round(2)
         .sort_values("Occupancy_Percentage", ascending=False))
print(fmt)

               Occupancy_Percentage  Tickets_Quantity  Ticket_Price  Concession_Spend
Screen_Format                                                                        
Standard 2D                   61.50              4.09        309.63            349.03
IMAX                          60.32              4.03        305.56            346.08
3D                            60.25              3.94        309.26            361.12


## Task 8. Ticket sales and concession spending: weekday vs weekend

In [12]:
wk = df.groupby("Day_Type").agg(
    Transactions=("Transaction_ID", "count"),
    Total_Tickets=("Tickets_Quantity", "sum"),
    Avg_Tickets_per_Txn=("Tickets_Quantity", "mean"),
    Ticket_Revenue=("Ticket_Revenue", "sum"),
    Total_Concession=("Concession_Spend", "sum"),
    Avg_Concession=("Concession_Spend", "mean"),
    Avg_Occupancy=("Occupancy_Percentage", "mean"),
).round(2)
print(wk)
# Per-day comparison (fairer: there are 5 weekdays vs 2 weekend days)
n_days = df.groupby("Day_Type")["Show_Date"].nunique()
print("\nDistinct calendar dates:\n", n_days)
print("\nAvg tickets sold per calendar date:\n",
      (wk["Total_Tickets"] / n_days).round(2))

          Transactions  Total_Tickets  Avg_Tickets_per_Txn  Ticket_Revenue  Total_Concession  Avg_Concession  Avg_Occupancy
Day_Type                                                                                                                   
Weekday            511           2051                 4.01       640400.22         177802.72          347.95          61.23
Weekend            189            764                 4.04       226810.77          68509.09          362.48          59.26

Distinct calendar dates:
 Day_Type
Weekday    309
Weekend    121
Name: Show_Date, dtype: int64

Avg tickets sold per calendar date:
 Day_Type
Weekday    6.64
Weekend    6.31
dtype: float64


## Task 9. Attendance across showtime slots

In [13]:
slots = df.groupby("Showtime_Slot").agg(
    Transactions=("Transaction_ID", "count"),
    Total_Tickets=("Tickets_Quantity", "sum"),
    Avg_Tickets=("Tickets_Quantity", "mean"),
    Avg_Occupancy=("Occupancy_Percentage", "mean"),
    Ticket_Revenue=("Ticket_Revenue", "sum"),
).round(2)
slots = slots.reindex(["Matinee", "Evening", "Late Night"])
print(slots)

               Transactions  Total_Tickets  Avg_Tickets  Avg_Occupancy  Ticket_Revenue
Showtime_Slot                                                                         
Matinee                 236            945         4.00          62.96       297496.22
Evening                 225            904         4.02          59.32       267988.52
Late Night              239            966         4.04          59.75       301726.25


## Task 10. Concession attach rate

In [14]:
df["Has_Concession"] = df["Concession_Spend"] > 0
attach_rate = df["Has_Concession"].mean() * 100
print(f"Overall attach rate: {attach_rate:.2f}%")
print("\nAttach rate by Day_Type (%):\n",
      (df.groupby("Day_Type")["Has_Concession"].mean() * 100).round(2))
print("\nAttach rate by Showtime_Slot (%):\n",
      (df.groupby("Showtime_Slot")["Has_Concession"].mean() * 100).round(2))
print("\nAttach rate by Screen_Format (%):\n",
      (df.groupby("Screen_Format")["Has_Concession"].mean() * 100).round(2))

Overall attach rate: 100.00%

Attach rate by Day_Type (%):
 Day_Type
Weekday    100.0
Weekend    100.0
Name: Has_Concession, dtype: float64

Attach rate by Showtime_Slot (%):
 Showtime_Slot
Evening       100.0
Late Night    100.0
Matinee       100.0
Name: Has_Concession, dtype: float64

Attach rate by Screen_Format (%):
 Screen_Format
3D             100.0
IMAX           100.0
Standard 2D    100.0
Name: Has_Concession, dtype: float64


## Task 11. Average ticket and snack spending by genre

In [15]:
genre = df.groupby("Genre").agg(
    Avg_Ticket_Price=("Ticket_Price", "mean"),
    Avg_Ticket_Revenue=("Ticket_Revenue", "mean"),
    Avg_Snack_Spend=("Concession_Spend", "mean"),
    Avg_Total_Spend=("Total_Spend", "mean"),
).round(2).sort_values("Avg_Total_Spend", ascending=False)
print(genre)

           Avg_Ticket_Price  Avg_Ticket_Revenue  Avg_Snack_Spend  Avg_Total_Spend
Genre                                                                            
Sci-Fi               314.57             1256.61           364.77          1621.38
Comedy               309.91             1252.54           354.47          1607.00
Action               303.97             1268.87           333.09          1601.96
Animation            296.02             1216.07           364.57          1580.64
Drama                315.03             1200.72           340.73          1541.46


## Task 12. NumPy: mean, median, std of total transaction spend

In [16]:
spend = df["Total_Spend"].to_numpy()
print(f"Mean   : {np.mean(spend):.2f}")
print(f"Median : {np.median(spend):.2f}")
print(f"Std Dev: {np.std(spend, ddof=1):.2f}  (sample, ddof=1)")

Mean   : 1590.75
Median : 1494.91
Std Dev: 772.77  (sample, ddof=1)


## Task 13. Unusually large group bookings using IQR on ticket quantity

In [17]:
q1, q3 = np.percentile(df["Tickets_Quantity"], [25, 75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Q1={q1}, Q3={q3}, IQR={iqr}, lower fence={lower}, upper fence={upper}")
df["Large_Group_Outlier"] = (df["Tickets_Quantity"] < lower) | (df["Tickets_Quantity"] > upper)
outliers = df[df["Large_Group_Outlier"]]
print("Outlier bookings found:", len(outliers))
if len(outliers):
    print(outliers[["Transaction_ID", "Movie_Title", "Tickets_Quantity"]].head(15))
else:
    print("No ticket quantity falls outside the IQR fences, so there are no "
          "statistical outliers (max group size is "
          f"{df['Tickets_Quantity'].max()}).")
    print("Largest bookings (top of the distribution):")
    print(df["Tickets_Quantity"].value_counts().sort_index(ascending=False).head(3))

Q1=2.0, Q3=6.0, IQR=4.0, lower fence=-4.0, upper fence=12.0
Outlier bookings found: 0
No ticket quantity falls outside the IQR fences, so there are no statistical outliers (max group size is 7).
Largest bookings (top of the distribution):
Tickets_Quantity
7    104
6    105
5    111
Name: count, dtype: int64


## Task 14. Ticket-quantity bands with pd.cut()

In [18]:
bins = [0, 1, 2, 4, np.inf]
labels = ["Solo", "Couple", "Small Group", "Large Group"]   # 1 | 2 | 3-4 | 5+
df["Group_Band"] = pd.cut(df["Tickets_Quantity"], bins=bins, labels=labels, right=True)
bands = df.groupby("Group_Band", observed=True).agg(
    Transactions=("Transaction_ID", "count"),
    Avg_Concession=("Concession_Spend", "mean"),
    Avg_Ticket_Revenue=("Ticket_Revenue", "mean"),
    Attach_Rate_Pct=("Has_Concession", lambda s: s.mean() * 100),
).round(2)
bands["Share_of_Txns_%"] = (bands["Transactions"] / bands["Transactions"].sum() * 100).round(1)
print(bands)

             Transactions  Avg_Concession  Avg_Ticket_Revenue  Attach_Rate_Pct  Share_of_Txns_%
Group_Band                                                                                     
Solo                  112          358.74              318.55            100.0             16.0
Couple                 93          346.98              595.32            100.0             13.3
Small Group           175          359.76             1053.99            100.0             25.0
Large Group           320          346.58             1849.12            100.0             45.7


## Task 15. Five marketing and scheduling recommendations (data-driven)

In [19]:
best_fmt = fmt["Occupancy_Percentage"].idxmax()
worst_fmt = fmt["Occupancy_Percentage"].idxmin()
best_slot = slots["Avg_Occupancy"].idxmax()
worst_slot = slots["Avg_Occupancy"].idxmin()
top_movie = movies["Ticket_Revenue"].idxmax()
top_genre_snack = genre["Avg_Snack_Spend"].idxmax()
low_genre_snack = genre["Avg_Snack_Spend"].idxmin()
best_band = bands["Avg_Concession"].idxmax()
wkend_conc = wk.loc["Weekend", "Avg_Concession"]
wkday_conc = wk.loc["Weekday", "Avg_Concession"]

recs = [
    f"1. Screen allocation: '{best_fmt}' has the highest average occupancy "
    f"({fmt.loc[best_fmt, 'Occupancy_Percentage']:.1f}%) while '{worst_fmt}' is lowest "
    f"({fmt.loc[worst_fmt, 'Occupancy_Percentage']:.1f}%). Give more screens/showtimes to "
    f"the former and use discounts or fill-the-house promos on the latter.",
    f"2. Scheduling: the '{best_slot}' slot fills best ({slots.loc[best_slot, 'Avg_Occupancy']:.1f}% "
    f"avg occupancy) and '{worst_slot}' fills worst "
    f"({slots.loc[worst_slot, 'Avg_Occupancy']:.1f}%). Put top titles in the strong slot; "
    f"use the weak slot for off-peak pricing or niche/re-run titles.",
    f"3. Top title: '{top_movie}' earns the most ticket revenue "
    f"({movies.loc[top_movie, 'Ticket_Revenue']:,.0f}). Give it prime screens and "
    f"IMAX/prime slots, and bundle its tie-in combos.",
    f"4. Combo promotions: '{top_genre_snack}' audiences spend the most on snacks "
    f"({genre.loc[top_genre_snack, 'Avg_Snack_Spend']:.0f}) and '{low_genre_snack}' the least "
    f"({genre.loc[low_genre_snack, 'Avg_Snack_Spend']:.0f}). Push premium combos for the first; "
    f"use ticket+snack bundles to lift spend for the second.",
    f"5. Group offers: the '{best_band}' band has the highest average concession spend. "
    f"Offer group combo packs (e.g. 4 tickets + shared snack platter). Weekend average snack "
    f"spend is {wkend_conc:.0f} vs {wkday_conc:.0f} on weekdays, so run weekday "
    f"snack-discount offers where the gap exists.",
]
for r in recs:
    print(r, "\n")

1. Screen allocation: 'Standard 2D' has the highest average occupancy (61.5%) while '3D' is lowest (60.2%). Give more screens/showtimes to the former and use discounts or fill-the-house promos on the latter. 

2. Scheduling: the 'Matinee' slot fills best (63.0% avg occupancy) and 'Evening' fills worst (59.3%). Put top titles in the strong slot; use the weak slot for off-peak pricing or niche/re-run titles. 

3. Top title: 'The Last Journey' earns the most ticket revenue (182,989). Give it prime screens and IMAX/prime slots, and bundle its tie-in combos. 

4. Combo promotions: 'Sci-Fi' audiences spend the most on snacks (365) and 'Action' the least (333). Push premium combos for the first; use ticket+snack bundles to lift spend for the second. 

5. Group offers: the 'Small Group' band has the highest average concession spend. Offer group combo packs (e.g. 4 tickets + shared snack platter). Weekend average snack spend is 362 vs 348 on weekdays, so run weekday snack-discount offers where 